# Step 1 : Load the Final Training Data

In [1]:
import pandas as pd

In [2]:
X_train = pd.read_csv("../Dataset/X_train.csv")

In [3]:
print(X_train.shape)

(7750, 189)


<img src="../images/Screenshot 2026-08-18 084148.png" alt="" width="300" height="200">

# Step : 2  Identify the Original User Inputs

In [4]:
print(X_train.columns[:10].tolist())

['total_sqft', 'bath', 'balcony', 'bhk', 'area_type_Carpet  Area', 'area_type_Plot  Area', 'area_type_Super built-up  Area', 'location_5th Phase JP Nagar', 'location_6th Phase JP Nagar', 'location_7th Phase JP Nagar']


In [5]:
print(X_train.columns[-10:].tolist())

['location_Varthur', 'location_Vidyaranyapura', 'location_Vijayanagar', 'location_Vittasandra', 'location_Whitefield', 'location_Yelachenahalli', 'location_Yelahanka', 'location_Yelahanka New Town', 'location_Yeshwanthpur', 'location_other']


### We need to preserve the exact same column order used during training. Otherwise, the ANN can receive the wrong feature in the wrong position and produce incorrect predictions.

# STEP 3 : Create Feature List for Flask

In [6]:
feature_columns = X_train.columns.tolist()
print("Total features:", len(feature_columns))

Total features: 189


## save this feature list:

In [7]:
import pickle

with open("../Models/feature_columns.pkl", "wb") as f:
    pickle.dump(feature_columns, f)

In [8]:
import os

print(os.path.exists("../Models/feature_columns.pkl"))

True


# Step 4 : Save/Verify the ANN Model

In [9]:
import os
model_path = "../Models/ann_model.keras"
print("Model exists:", os.path.exists(model_path))

Model exists: True


<img src="../images/Screenshot 2026-08-18 085602.png" alt="" width="300" height="200">

# Step 5 : Verify the 6 User Inputs

### Before creating the Flask form, we need to confirm which original columns should be entered by the user

In [10]:
area_type_columns = [
    col for col in feature_columns
    if col.startswith("area_type_")
]

location_columns = [
    col for col in feature_columns
    if col.startswith("location_")
]

print("Area Types:")
print(area_type_columns)

print("\nNumber of locations:", len(location_columns))

print("\nFirst 10 locations:")
print(location_columns[:10])

print("\nLast 10 locations:")
print(location_columns[-10:])

Area Types:
['area_type_Carpet  Area', 'area_type_Plot  Area', 'area_type_Super built-up  Area']

Number of locations: 182

First 10 locations:
['location_5th Phase JP Nagar', 'location_6th Phase JP Nagar', 'location_7th Phase JP Nagar', 'location_8th Phase JP Nagar', 'location_9th Phase JP Nagar', 'location_Abbigere', 'location_Akshaya Nagar', 'location_Ambalipura', 'location_Ambedkar Nagar', 'location_Amruthahalli']

Last 10 locations:
['location_Varthur', 'location_Vidyaranyapura', 'location_Vijayanagar', 'location_Vittasandra', 'location_Whitefield', 'location_Yelachenahalli', 'location_Yelahanka', 'location_Yelahanka New Town', 'location_Yeshwanthpur', 'location_other']


## So total 6 input like this:

<img src="..\images\Screenshot 2026-08-18 090412.png" alt="" width="300" height="200">

# STEP 6 : Create 189-feature input list

In [11]:
input_data = [0] * len(feature_columns)

print(len(input_data))

189


In [12]:
total_sqft = 1200
bath = 2
balcony = 1
bhk = 2

In [13]:
input_data[0] = total_sqft
input_data[1] = bath
input_data[2] = balcony
input_data[3] = bhk

In [14]:
area_type = "Super built-up  Area"

In [15]:
area_column = "area_type_" + area_type

In [16]:
print(area_column)

area_type_Super built-up  Area


### 6.1 — Find its position

In [17]:
print(feature_columns[:7])

['total_sqft', 'bath', 'balcony', 'bhk', 'area_type_Carpet  Area', 'area_type_Plot  Area', 'area_type_Super built-up  Area']


In [18]:
#input_data[area_index] = 1
area_index = feature_columns.index(area_column)
#area_index = 0  # or any valid index
input_data[area_index] = 1


In [19]:
location = "Whitefield"

In [20]:
location_column = "location_" + location

In [21]:
print(location_column)

location_Whitefield


### 6.2 — Find Location Position

In [22]:
location_index = feature_columns.index(location_column)
print(location_index)

183


In [23]:
input_data[location_index] = 1

In [24]:
print(len(input_data))

189


In [25]:
print(input_data[:10])

[1200, 2, 1, 2, 0, 0, 1, 0, 0, 0]


# Step 7 — Load the ANN Model

In [26]:
# !pip install tensorflow

In [27]:
from tensorflow.keras.models import load_model

In [28]:
model = load_model("../Models/ann_model.keras")
print(model.input_shape)

(None, 189)


# Step 8 : Make Prediction

In [29]:
import numpy as np

In [30]:
input_array = np.array(input_data).reshape(1, -1)
print(input_array.shape)

(1, 189)


###  8.1 — Predict

In [31]:
prediction = model.predict(input_array)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step


In [32]:
print(prediction)

[[50909.066]]


### The ANN expects the scaled version of those 189 features.

<img src="..\images\Screenshot 2026-08-18 095350.png" alt="" width="300" height="200">

# Step 9 — Find Your Scaler

In [33]:
import os
print(os.path.exists("../Models/scaler.pkl"))

True


## 9.1 — Load the Original Scaler

In [34]:
import pickle

In [35]:
with open("../Models/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

In [36]:
print(scaler)

StandardScaler()


# Step 10 — Scale the Test Input

In [37]:
#input_scaled = scaler.transform([input_data])

import pandas as pd

input_df = pd.DataFrame([input_data], columns=X_train.columns)
input_scaled = scaler.transform(input_df)

In [38]:
print(input_scaled.shape)

(1, 189)


### The ANN was trained using the scaled version, so now we're giving it data in the same format.

# Step 11 :  Make the Correct ANN Prediction

### 11.1 — Predict

In [39]:
prediction = model.predict(input_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step


### 11.2 — Get the predicted price

In [40]:
predicted_price = prediction[0][0]
print("Predicted Price:", predicted_price)

Predicted Price: 55.50807


### 11.3 — Convert to a more user-friendly format

In [41]:
print(f"Predicted Price: ₹{predicted_price:.2f} Lakhs")

Predicted Price: ₹55.51 Lakhs


## Current pipeline

<img src="..\images\Screenshot 2026-08-18 100509.png" alt="" width="300" height="200">

# Step 13 — Make the Test Prediction Reusable

In [42]:
def predict_price(total_sqft, bath, balcony, bhk, area_type, location):
    input_data = [0] * len(feature_columns)
    
    input_data[0] = total_sqft
    input_data[1] = bath
    input_data[2] = balcony
    input_data[3] = bhk

    area_column = "area_type_" + area_type
    area_index = feature_columns.index(area_column)
    input_data[area_index] = 1

    location_column = "location_" + location

    if location_column in feature_columns:
        location_index = feature_columns.index(location_column)
        input_data[location_index] = 1
    else:
        input_data[feature_columns.index("location_other")] = 1

    input_df = pd.DataFrame([input_data], columns=feature_columns)
    input_scaled = scaler.transform(input_df)

    prediction = model.predict(input_scaled, verbose=0)

    return prediction[0][0]

## 13.1 Function test

In [43]:
test_price = predict_price(
    1200,
    2,
    1,
    2,
    "Super built-up  Area",
    "Whitefield"
)

In [44]:
print(f"Predicted Price: ₹{test_price:.2f} Lakhs")

Predicted Price: ₹55.51 Lakhs


## 13.2 : Second test

In [45]:
test_price_2 = predict_price(
    1500,
    3,
    2,
    3,
    "Super built-up  Area",
    "Yeshwanthpur"
)

In [46]:
print(f"Predicted Price: ₹{test_price_2:.2f} Lakhs")

Predicted Price: ₹98.32 Lakhs


## 13.3 : Third Test

In [47]:
test_price_3 = predict_price(
    1000,
    2,
    1,
    2,
    "Super built-up  Area",
    "other"
)

In [48]:
print(f"Predicted Price: ₹{test_price_3:.2f} Lakhs")

Predicted Price: ₹42.81 Lakhs


In [49]:
print("area_type =", repr(area_type))
print("area_column =", repr(area_column))
print("feature_columns =")
print(feature_columns)

area_type = 'Super built-up  Area'
area_column = 'area_type_Super built-up  Area'
feature_columns =
['total_sqft', 'bath', 'balcony', 'bhk', 'area_type_Carpet  Area', 'area_type_Plot  Area', 'area_type_Super built-up  Area', 'location_5th Phase JP Nagar', 'location_6th Phase JP Nagar', 'location_7th Phase JP Nagar', 'location_8th Phase JP Nagar', 'location_9th Phase JP Nagar', 'location_Abbigere', 'location_Akshaya Nagar', 'location_Ambalipura', 'location_Ambedkar Nagar', 'location_Amruthahalli', 'location_Anandapura', 'location_Ananth Nagar', 'location_Anekal', 'location_Anjanapura', 'location_Ardendale', 'location_Arekere', 'location_Attibele', 'location_BTM 2nd Stage', 'location_BTM Layout', 'location_Babusapalaya', 'location_Balagere', 'location_Banashankari', 'location_Banashankari Stage II', 'location_Banashankari Stage III', 'location_Banashankari Stage V', 'location_Banaswadi', 'location_Bannerghatta Road', 'location_Basavangudi', 'location_Basaveshwara Nagar', 'location_Battar

In [50]:
# print(repr(area_type))
# print(repr(area_column))
# print(repr(feature_columns[feature_columns.index(area_column)]))


matches = [col for col in feature_columns if 'Built-up' in col]

print("Matching columns:", matches)

Matching columns: []


In [51]:
area_type = "Super built-up  Area"

area_column = "area_type_" + area_type

print(repr(area_type))
print(repr(area_column))
print(area_column in feature_columns)

'Super built-up  Area'
'area_type_Super built-up  Area'
True
